# A5 sinh số lượng (TASKS "A5, bước tiếp theo (10/11)", bước 1 đến 3). CHƯA train gì.
Sinh 4000 mẫu call bằng mã đã sửa → **Qwen3-8B và Qwen3.5-9B viết câu** (mỗi model một nửa, không tự chấm) → **khứ hồi** bằng Qwen3-4B + `q4b_lora` (3 tiến trình song song) → tổng hợp → xem trước số mẫu dùng được theo bộ chia A4. Khoảng **1,5 đến 2,5 giờ GPU** (sinh call vài phút, viết câu khoảng 20 đến 30 phút, khứ hồi 45 đến 90 phút).

**Trước khi chạy cell đầu tiên:** đặt notebook **Private**, Internet **Off**, Accelerator **GPU (RTX PRO 6000)**.
**Input cần gắn (6 thứ):**
1. dataset `lifemate-code` **bản mới từ commit `b2853d2`** (có `soft_failure`, sửa `--weights`, giờ quá khứ, lệnh ghi vô hiệu, bộ lọc tên cũ `stale_name`; cell A-1 chặn nếu bản cũ)
2. dataset `train+public_test` (raw)
3. dataset `thuan-qwen3-4b`
4. **output của notebook `Hello_done`** (có `state/processed` và adapter `q4b_lora`)
5. Model **Qwen3-8B**
6. Model **Qwen3.5-9B**

**KHÔNG gắn** output A4, A4b, A5 cũ hay `train-hist` (cùng có `state/processed`, làm bước khôi phục chọn nhầm). Mẫu sinh từ mã cũ (các lần thử trước) **bỏ hết**, notebook này sinh lại từ đầu. Câu nói và call là dữ liệu cuộc thi: notebook và output để **Private**, zip chỉ gửi qua Drive riêng.

**Thứ tự cell:** P-1 đến P-3, A-1 (kiểm tra), A-2 (tìm hai model, sha256), A-3 (sinh call), A-4-8b, A-4-9b (viết câu), A-5 (khứ hồi, lâu nhất), A-6 (tổng hợp), A-7 (xem trước build_sft), A-8 (đóng zip), A-9 (dọn, chạy cuối rồi Save Version).
**Gửi Phương TRƯỚC KHI train:** `a5_gen_results.zip` (qua Drive riêng) và dòng `callgen -> sft_train_cg.jsonl: N samples; {...}` ở A-7, cùng số mẫu được giữ theo số call ở A-6. Phương duyệt 30 dòng rồi mới chạy notebook train.
**Chạy lại được:** A-4 và A-5 tự tiếp tục từ chỗ dừng nếu bị ngắt. Output của notebook này chứa `a5_data/callgen_checked_all.jsonl` cho notebook train; **không có thư mục `state/`** (cố ý).

In [ ]:
%%bash
# P-1: setup. Dùng kaggle_setup.sh của Phương; chỉ đặt đường dẫn model cho khớp đúng thứ ta gắn:
#  - QWEN = Qwen3-4B (hidden 2560, 36 tầng), tìm theo cấu hình vì còn có Qwen3-8B/Qwen3.5-9B trong input (không lấy "qwen3 đầu tiên")
#  - PHO = đường dẫn giả (A5 không dùng PhoWhisper)
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
find /kaggle/input -maxdepth 8 -name config.json 2>/dev/null > /kaggle/working/config_paths.txt
QW=$(python -X utf8 - <<'EOF'
import json
from pathlib import Path
for line in open("/kaggle/working/config_paths.txt", encoding="utf-8"):
    p = Path(line.strip())
    try:
        c = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue
    t = c.get("text_config", c)
    if c.get("model_type") == "qwen3" and t.get("hidden_size") == 2560 and t.get("num_hidden_layers") == 36:
        print(p.parent)
        break
EOF
)
echo "Qwen3-4B dir: ${QW:-KHÔNG THẤY}"
[ -n "$QW" ] || { echo "ERROR: không thấy Qwen3-4B (hidden 2560, 36 tầng). Gắn dataset thuan-qwen3-4b."; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e 's#^PHO=.*#PHO=/nonexistent/phowhisper#' -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục từ output notebook trước? (checkpoints và outputs)"; ls /kaggle/working/checkpoints | head -20; ls /kaggle/working/outputs | head -12

In [ ]:
%%bash
# P-2: in phiên bản torch/CUDA/GPU, kiểm tra GPU, peft, accelerate và việc transformers nhận qwen3_5
pip uninstall -y torchao 2>&1 | tail -1
python -X utf8 - <<'EOF' | tee /kaggle/working/torch_cuda.txt
import sys
import torch
print(torch.__version__, torch.version.cuda, sys.version)
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__, "| gpus", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("transformers nhận qwen3:", "qwen3" in CONFIG_MAPPING, "| qwen3_5 (Qwen3.5):", "qwen3_5" in CONFIG_MAPPING)
try:
    import accelerate
    print("accelerate", accelerate.__version__)
except ImportError:
    print("THIẾU accelerate: gen_text.py sẽ lỗi ở bước nạp model (device_map). Báo lại, đừng pip install từ mạng.")
if not torch.cuda.is_available():
    raise SystemExit("Phiên này KHÔNG có GPU: bật Accelerator rồi chạy lại")
print("VRAM tổng: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
EOF

In [ ]:
%%bash
# P-3: ghi sẵn hai đoạn mã phụ (chỉ in số đếm tổng hợp, không in nội dung câu nói hay call)
mkdir -p /kaggle/working/helpers
cat > /kaggle/working/helpers/a5g_summary.py <<'PYEOF'
"""Tổng hợp A5 sinh số lượng: chỉ in SỐ ĐẾM (không in nội dung câu nói hay call). Ghi 30 dòng ngẫu nhiên của mẫu ĐƯỢC GIỮ ra file để Phương duyệt.
usage: python a5g_summary.py <callgen_checked_all.jsonl> <txt_8b.jsonl> <txt_9b.jsonl> <out_dir>
Chạy từ thư mục gốc repo (cần data/raw/..., src/)."""
import json
import random
import re
import statistics
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from lifemate.callgen import app_of, text_ok  # noqa: E402
from lifemate.jsonl import read_jsonl  # noqa: E402
from lifemate.leakcheck import normalize  # noqa: E402
from lifemate.paths import split_dir  # noqa: E402
from lifemate.prompt import load_catalog  # noqa: E402

HELD = {"flights", "bike_rental", "package_tracking", "shopping_list", "podcast_radio", "loyalty"}
checked, f8, f9, out_dir = Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3]), Path(sys.argv[4])
out_dir.mkdir(parents=True, exist_ok=True)
rows = read_jsonl(checked)
writer = {}
for tag, f in (("8b", f8), ("9b", f9)):
    if f.is_file():
        for r in read_jsonl(f):
            writer[r["id"]] = tag
cat = load_catalog(split_dir("train") / "train_tools_catalog.json")
train = read_jsonl(split_dir("train") / "train.jsonl")
real_norm = {normalize(it["audio_text"]) for it in train}
cjk = re.compile(r"[぀-ヿ㐀-鿿가-힯]")


def pct(a, b):
    return f"{a}/{b} ({100 * a / b:.1f}%)" if b else f"{a}/0"


n = len(rows)
for r in rows:
    r["_ok"], r["_why"] = text_ok(r["audio_text"], r, cat)
    r["_w"] = writer.get(r["id"], "?")
    r["_nc"] = len(r["label"]["tool_calls"])
    r["_copy"] = normalize(r["audio_text"]) in real_norm
    r["_apps"] = {app_of(c["name"]) for c in r["label"]["tool_calls"]}
    r["_held"] = bool(r["_apps"] & HELD)
kept = [r for r in rows if r.get("roundtrip")]
print(f"== A5 sinh số lượng: {n} mẫu trong {checked.name}")
print(f"qua luật text_ok                          : {pct(sum(r['_ok'] for r in rows), n)}")
print(f"qua bộ lọc rẻ (prefilter_ok)              : {pct(sum(bool(r.get('prefilter_ok')) for r in rows), n)}")
pre_rows = [r for r in rows if r.get("prefilter_ok")]
print(f"khứ hồi đạt trong mẫu qua lọc rẻ (agent) : {pct(sum(bool(r.get('agent_ok')) for r in pre_rows), len(pre_rows))}")
print(f"ĐƯỢC GIỮ (roundtrip = lọc rẻ VÀ khứ hồi)  : {pct(len(kept), n)}")
print("lý do text_ok loại:", dict(Counter(r["_why"] for r in rows).most_common()))

print("\n-- theo model viết câu (tổng | qua lọc rẻ | được giữ):")
for w in sorted({r["_w"] for r in rows}):
    g = [r for r in rows if r["_w"] == w]
    print(f"  {w}: {len(g):4} | {sum(bool(r.get('prefilter_ok')) for r in g):4} | {sum(bool(r.get('roundtrip')) for r in g):4}"
          f"   (khứ hồi đạt trong mẫu qua lọc: {pct(sum(bool(r.get('agent_ok')) for r in g if r.get('prefilter_ok')), sum(bool(r.get('prefilter_ok')) for r in g))})")

print("\n-- theo số call (tổng | qua lọc rẻ | được giữ | % giữ):")
for k in sorted({r["_nc"] for r in rows}):
    g = [r for r in rows if r["_nc"] == k]
    kk = sum(bool(r.get("roundtrip")) for r in g)
    print(f"  {k} call: {len(g):4} | {sum(bool(r.get('prefilter_ok')) for r in g):4} | {kk:4} | {100 * kk / len(g):.0f}%")
print("\n-- theo số APP trong mẫu (tổng | được giữ):")
for k in sorted({len(r["_apps"]) for r in rows}):
    g = [r for r in rows if len(r["_apps"]) == k]
    print(f"  {k} app: {len(g):4} | {sum(bool(r.get('roundtrip')) for r in g):4}")

words = sorted(len(r["audio_text"].split()) for r in kept) or [0]
print(f"\nmẫu được giữ: số từ trung vị {statistics.median(words)}, p90 {words[int(0.9 * (len(words) - 1))]}, max {words[-1]}")
print(f"câu rỗng: {sum(1 for r in rows if not r['audio_text'].strip())} | có ký tự Hán/Nhật/Hàn: {sum(bool(cjk.search(r['audio_text'])) for r in rows)} "
      f"(trong số được giữ: {sum(bool(cjk.search(r['audio_text'])) for r in kept)})")
print(f"câu trùng y hệt câu THẬT: {pct(sum(r['_copy'] for r in rows), n)} (trong số được giữ: {sum(r['_copy'] for r in kept)}; build_sft sẽ bỏ)")
print(f"câu trùng nhau trong số được giữ: {len(kept) - len({normalize(r['audio_text']) for r in kept})}")
print(f"mẫu dùng 1 trong 6 app A4 giấu: {pct(sum(r['_held'] for r in rows), n)} (trong số được giữ: {sum(r['_held'] for r in kept)}; bị bỏ khi train theo bộ chia A4)")
usable_full = [r for r in kept if not r["_copy"] and r["_ok"]]
usable_a4 = [r for r in usable_full if not r["_held"]]
print(f"ƯỚC LƯỢNG số mẫu dùng được: train toàn bộ {len(usable_full)}; theo bộ chia A4 tối đa {len(usable_a4)} "
      f"(chưa trừ mẫu có mẫu gốc/donor nằm ngoài train của bộ chia A4; số chính xác ở cell xem trước build_sft)")
print("số app mỗi mẫu được giữ:", dict(sorted(Counter(len(r["_apps"]) for r in kept).items())))
print("app nhiều nhất trong số được giữ:", dict(Counter(a for r in kept for a in r["_apps"]).most_common(8)))

# 30 dòng ngẫu nhiên của mẫu ĐƯỢC GIỮ (seed 0), kèm mục id trỏ tới
pool = sorted(kept, key=lambda r: r["id"])
pick = random.Random(0).sample(pool, min(30, len(pool)))
with open(out_dir / "a5g_sample30_kept.jsonl", "w", encoding="utf-8", newline="\n") as f:
    for r in pick:
        f.write(json.dumps({"id": r["id"], "writer": r["_w"], "audio_text": r["audio_text"], "tool_calls": r["label"]["tool_calls"],
                            "refs": r.get("refs", {}), "text_ok": r["_ok"], "agent_ok": r.get("agent_ok")}, ensure_ascii=False) + "\n")
with open(out_dir / "a5g_sample30_kept.txt", "w", encoding="utf-8", newline="\n") as f:
    for i, r in enumerate(pick, 1):
        f.write(f"#{i} id={r['id']} writer={r['_w']} text_ok={r['_ok']} agent_ok={r.get('agent_ok')}\n  CÂU : {r['audio_text']}\n")
        for c in r["label"]["tool_calls"]:
            f.write(f"  CALL: {json.dumps(c, ensure_ascii=False)}\n")
        for k, v in (r.get("refs") or {}).items():
            f.write(f"  REF : {k} -> {json.dumps(v, ensure_ascii=False)[:240]}\n")
        f.write("\n")
print(f"\nđã ghi 30 dòng ngẫu nhiên của mẫu được giữ (seed 0) -> {out_dir}/a5g_sample30_kept.txt và .jsonl (KHÔNG in nội dung ở đây)")
PYEOF
cat > /kaggle/working/helpers/pack_zip.py <<'PYEOF'
"""Đóng gói file kết quả nhỏ thành một zip kèm MANIFEST sha256 (không đóng gói trọng số).
usage: python pack_zip.py <out.zip> <file> [<file> ...]   (file không tồn tại thì bỏ qua và báo)"""
import hashlib
import sys
import zipfile
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


out = Path(sys.argv[1])
files = []
for name in sys.argv[2:]:
    p = Path(name)
    if p.is_file():
        files.append(p)
    else:
        print("bỏ qua (không có):", name)
files = sorted(set(files), key=str)
manifest = "".join(f"{sha256(p)}  {p.as_posix()}  ({p.stat().st_size} bytes)\n" for p in files)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.as_posix().lstrip("/"))
    z.writestr("MANIFEST_sha256.txt", manifest)
print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(files)} file")
PYEOF
ls -la /kaggle/working/helpers

In [ ]:
%%bash
# A-1: kiểm tra MÃ MỚI (từ commit b2853d2), adapter q4b_lora (người kiểm tra khứ hồi) và dev_ids. Thiếu hoặc sai thì DỪNG.
# Rồi dọn bản sao các adapter không dùng mà setup đã chép từ Hello_done, và xóa tệp trung gian cũ (nếu có).
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
MISSING=0
need() { if [ -s "$1" ]; then echo "ok     $1"; else echo "THIẾU  $1"; MISSING=1; fi; }
need scripts/gen_calls.py
need scripts/gen_text.py
need scripts/verify_callgen.py
need scripts/build_sft.py
need src/lifemate/callgen.py
need src/lifemate/leakcheck.py
need data/processed/dev_ids.txt
need data/processed/asr_phowl_train.jsonl
need checkpoints/q4b_lora/adapter_model.safetensors
need checkpoints/q4b_lora/adapter_config.json
need models.lock.json
need "data/raw/train+public_test/train/train.jsonl"
grep -q "soft_failure" src/lifemate/callgen.py 2>/dev/null && echo "ok     code là bản MỚI (có soft_failure)" || { echo "THIẾU  code còn là bản CŨ: upload lifemate-code mới (từ commit b2853d2)"; MISSING=1; }
grep -q "def resolve_refs" src/lifemate/callgen.py 2>/dev/null || { echo "THIẾU  resolve_refs: code cũ"; MISSING=1; }
grep -q "def stale_name" src/lifemate/callgen.py 2>/dev/null && echo "ok     có stale_name (bộ lọc tên cũ của mẫu cho)" || { echo "THIẾU  stale_name: code cũ (cần b2853d2)"; MISSING=1; }
grep -q -- "--no-verify" scripts/gen_text.py 2>/dev/null || { echo "THIẾU  gen_text chưa có --no-verify: code cũ"; MISSING=1; }
if [ "$MISSING" = 1 ]; then echo "-> Kiểm tra input: lifemate-code (bản từ b2853d2), train+public_test, thuan-qwen3-4b, output notebook Hello_done."; exit 1; fi
n=$(wc -l < data/processed/dev_ids.txt)
[ "$n" = 155 ] || { echo "dev_ids.txt có $n dòng, phải là 155"; exit 1; }
echo "dev_ids.txt: 155 id (đúng)"
h=$(sha256sum checkpoints/q4b_lora/adapter_model.safetensors | cut -d' ' -f1)
want=$(grep -o '"adapter_model.safetensors": *"[0-9a-f]*"' models.lock.json | grep -o '[0-9a-f]\{64\}' | head -1)
echo "q4b_lora sha256 : $h"; echo "models.lock.json: $want"
if [ "$h" = "$want" ]; then echo "q4b_lora: TRÙNG adapter của bài nộp"; else echo "CẢNH BÁO: q4b_lora KHÁC adapter ghi trong models.lock.json. Báo Phương."; fi
echo "-- dọn bản sao không dùng và tệp trung gian cũ"
for d in checkpoints/*; do [ "$(basename "$d")" = "q4b_lora" ] || rm -rf "$d"; done
rm -rf checkpoints/q4b_lora/checkpoint-*
rm -rf /kaggle/working/outputs/*
mkdir -p outputs/logs
rm -f data/processed/callgen_* data/processed/pend_* data/processed/txt_* data/processed/vpart_* data/processed/chk_*
rm -rf data/processed/holdout
ls checkpoints; df -h /kaggle/working | tail -1

In [ ]:
%%bash
# A-2: tìm cả hai model viết câu trong /kaggle/input theo cấu hình (không đoán tên thư mục) và đối chiếu sha256 với bản gốc Hugging Face.
# Mất khoảng 3 đến 6 phút (đọc 36 GB để tính sha256). Chỉ thử một model: đặt GEN_SIZES="8b" (hoặc "9b") ở dòng dưới.
export GEN_SIZES=${GEN_SIZES:-"8b 9b"}
find /kaggle/input -maxdepth 8 -name config.json 2>/dev/null > /kaggle/working/config_paths.txt
python -X utf8 - <<'EOF'
import hashlib
import json
import os
import sys
from pathlib import Path

# (model_type, hidden_size, số tầng văn bản). Qwen3.5 là model đa phương thức: cấu hình văn bản nằm trong "text_config".
WANT = {"8b": ("qwen3", 4096, 36), "9b": ("qwen3_5", 4096, 32), "14b": ("qwen3", 5120, 40), "32b": ("qwen3", 5120, 64)}
# sha256 các shard của bản gốc trên Hugging Face (Qwen/Qwen3-8B và Qwen/Qwen3.5-9B), tra theo kích thước file vì Kaggle có thể đổi tên shard.
OFFICIAL = {
    "8b": {3996250744: "31d6a825ae35f11fb85b195b4c42c146c051e446433125a215336abdf95cbf5f",
           3993160032: "5991236cea6fe21f3d43cab0f0e84448734fbbe0789816202989f2ddc9d18282",
           3959604768: "c5185c4794be2d8a9784d5753c9922db38df478ce11f9ed0b415b7304d896836",
           3187841392: "b5ee7de71fbf17db3d5704e0c8f2bc7d005ca9e1d7ca2aeb19827b0cfcaa917a",
           1244659840: "20c2d6366ab85c90786ccdd829cd2b9e7d30ef3b2ebbb998280e7e4014b542ff"},
    "9b": {5276436216: "db6f444b43d318c92f360a13a25561a6a65b10c0631b8ed305a426dbaa6c380e",
           5335161512: "31c7d7e2dd5d207840b31cc59083c8f4c4718959149e0358c0364052bb9a0330",
           5368717440: "7ec36ba3a4176a44c3c0876ad80c56a2f70c84bf008d82e9501df642f17dadec",
           3325995712: "b62b0c4cd7e44edee103ee8f4fe225f246d5e768e07bfd5f25b63a8aa1fdd0c6"},
}
sizes = os.environ["GEN_SIZES"].split()
found = []
for line in open("/kaggle/working/config_paths.txt", encoding="utf-8"):
    p = Path(line.strip())
    try:
        c = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue
    if not str(c.get("model_type", "")).startswith("qwen3"):
        continue
    t = c.get("text_config", c)
    weights = sorted(p.parent.glob("*.safetensors"))
    found.append((p.parent, c.get("model_type"), t.get("hidden_size"), t.get("num_hidden_layers"), len(weights),
                  sum(w.stat().st_size for w in weights) / 1e9))
print("Các model qwen3* đang gắn vào notebook:")
for d, mt, h, L, nw, gb in found:
    print(f"  {d}  type={mt} hidden={h} layers={L}  {nw} file safetensors, {gb:.1f} GB")

env_lines, info = [], []
problems = []
for size in sizes:
    want = WANT[size]
    pick = [f for f in found if (f[1], f[2], f[3]) == want and f[4] > 0]
    if not pick:
        problems.append(f"KHÔNG thấy model {size} (type={want[0]}, hidden={want[1]}, layers={want[2]}) có safetensors. "
                        "Add Input > Models > tìm đúng model rồi chạy lại cell này.")
        continue
    d = pick[0][0]
    env_lines.append(f'export GEN_{size.upper()}="{d}"')
    info.append(f"== GEN_SIZE={size}\nthư mục: {d}\nconfig.json sha256: {hashlib.sha256((d / 'config.json').read_bytes()).hexdigest()}\n"
                f"model_type={pick[0][1]} hidden_size={pick[0][2]} num_hidden_layers={pick[0][3]} tổng safetensors={pick[0][5]:.2f} GB")
    ref = OFFICIAL.get(size)
    if ref:
        verdict, matched = [], set()
        shards = sorted(d.glob("*.safetensors"))
        for w in shards:
            sz = w.stat().st_size
            if sz not in ref:
                verdict.append(f"{w.name}: kích thước KHÁC bản gốc (không đối chiếu được)")
                continue
            h = hashlib.sha256()
            with open(w, "rb") as fh:
                while block := fh.read(1 << 24):
                    h.update(block)
            ok = h.hexdigest() == ref[sz]
            if ok:
                matched.add(sz)
            verdict.append(f"{w.name}: " + ("TRÙNG bản gốc" if ok else "KHÁC bản gốc (sha256 không trùng)"))
        full = matched == set(ref) and len(shards) == len(ref)
        head = (f"ĐỐI CHIẾU với bản gốc Hugging Face: {len(matched)}/{len(ref)} shard trùng sha256 (đủ, là bản gốc)" if full else
                f"ĐỐI CHIẾU với bản gốc Hugging Face: CHƯA XÁC MINH ({len(matched)}/{len(ref)} shard trùng)")
        info.append(head + "\n" + "\n".join("  " + v for v in verdict))
        print(f"{size}: {head}")
    print(f"{size}: DÙNG {d}")
if problems:
    sys.exit("\n".join(problems))
Path("/kaggle/working/a5_env.sh").write_text("\n".join(env_lines) + f'\nexport GEN_SIZES="{" ".join(sizes)}"\n', encoding="utf-8")
Path("/kaggle/working/gen_model_info.txt").write_text("\n\n".join(info) + "\n", encoding="utf-8")
EOF
cat /kaggle/working/a5_env.sh

In [ ]:
%%bash
# A-3 (CPU, khoảng 1 đến 3 phút): sinh 4000 mẫu call bằng MÃ MỚI, rồi chia đôi cho hai model viết câu (bước 1 của Phương).
# --weights = số ĐOẠN trộn vào một mẫu (theo phân bố thật của train). Seed 42 cố định: chạy lại ra đúng cùng tệp (an toàn để chạy lại).
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 scripts/gen_calls.py --n 4000 --seed 42 --weights 1:0.35,2:0.4,3:0.2,4:0.05 2>&1 | tee outputs/logs/a5g_gen_calls.txt
wc -l data/processed/callgen_pending.jsonl
split -n l/2 -d data/processed/callgen_pending.jsonl data/processed/pend_
wc -l data/processed/pend_00 data/processed/pend_01

In [ ]:
%%bash
# A-4-8b (GPU, khoảng 8 đến 15 phút): model 8b viết câu cho một nửa số mẫu, KHÔNG tự chấm (--no-verify, theo Phương). Chạy trên máy Kaggle, không gọi API ngoài.
# Chạy lại cell thì tự tiếp tục (gen_text bỏ qua dòng đã có). Nếu lỗi nạp model: báo NGUYÊN VĂN dòng lỗi, không tự sửa mã.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/a5_env.sh
SIZE=8b
VAR=GEN_${SIZE^^}
GEN=${!VAR}
echo "model: $GEN ($SIZE), đầu vào: data/processed/pend_00"
START=$(date +%s)
python -X utf8 scripts/gen_text.py --model "$GEN" --inp data/processed/pend_00 --out data/processed/txt_${SIZE}.jsonl --batch 32 --no-verify 2>&1 | tee -a outputs/logs/a5g_gen_text_${SIZE}.txt | grep -v "Loading weights" | tail -n 5
echo "thời gian gen_text: $(( $(date +%s) - START )) s (gồm nạp model)" | tee -a outputs/logs/a5g_gen_text_${SIZE}.txt
wc -l data/processed/pend_00 data/processed/txt_${SIZE}.jsonl

In [ ]:
%%bash
# A-4-9b (GPU, khoảng 8 đến 15 phút): model 9b viết câu cho một nửa số mẫu, KHÔNG tự chấm (--no-verify, theo Phương). Chạy trên máy Kaggle, không gọi API ngoài.
# Chạy lại cell thì tự tiếp tục (gen_text bỏ qua dòng đã có). Nếu lỗi nạp model: báo NGUYÊN VĂN dòng lỗi, không tự sửa mã.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
source /kaggle/working/a5_env.sh
SIZE=9b
VAR=GEN_${SIZE^^}
GEN=${!VAR}
echo "model: $GEN ($SIZE), đầu vào: data/processed/pend_01"
START=$(date +%s)
python -X utf8 scripts/gen_text.py --model "$GEN" --inp data/processed/pend_01 --out data/processed/txt_${SIZE}.jsonl --batch 32 --no-verify 2>&1 | tee -a outputs/logs/a5g_gen_text_${SIZE}.txt | grep -v "Loading weights" | tail -n 5
echo "thời gian gen_text: $(( $(date +%s) - START )) s (gồm nạp model)" | tee -a outputs/logs/a5g_gen_text_${SIZE}.txt
wc -l data/processed/pend_01 data/processed/txt_${SIZE}.jsonl

In [ ]:
%%bash
# A-5 (GPU, khoảng 45 đến 90 phút): khứ hồi bằng Qwen3-4B + adapter q4b_lora, 3 TIẾN TRÌNH SONG SONG trên cùng GPU (bước 3 của Phương), bỏ --check-all.
# Chạy lại cell thì tự tiếp tục (verify_callgen bỏ qua dòng đã có trong chk_*). Log từng tiến trình: outputs/logs/a5g_verify_0*.txt
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
for f in data/processed/txt_8b.jsonl data/processed/txt_9b.jsonl; do [ -s "$f" ] || { echo "THIẾU $f: chạy cell A-4 cho cả hai model trước"; exit 1; }; done
cat data/processed/txt_8b.jsonl data/processed/txt_9b.jsonl > data/processed/txt_all.jsonl
wc -l data/processed/txt_all.jsonl
split -n l/3 -d data/processed/txt_all.jsonl data/processed/vpart_
wc -l data/processed/vpart_0*
START=$(date +%s)
pids=()
for k in 00 01 02; do
  python -X utf8 scripts/verify_callgen.py --model "$QWEN" --adapter checkpoints/q4b_lora --inp data/processed/vpart_$k --out data/processed/chk_$k.jsonl \
    >> outputs/logs/a5g_verify_$k.txt 2>&1 &
  pids+=($!)
done
fail=0
for p in "${pids[@]}"; do wait "$p" || fail=1; done
echo "thời gian khứ hồi: $(( $(date +%s) - START )) s"
for k in 00 01 02; do echo "-- tiến trình $k:"; tail -n 3 outputs/logs/a5g_verify_$k.txt | cut -c1-200; done
wc -l data/processed/chk_0*.jsonl
[ "$fail" = 0 ] || { echo "MỘT TIẾN TRÌNH LỖI: xem outputs/logs/a5g_verify_0*.txt (báo NGUYÊN VĂN dòng lỗi); chạy lại cell sẽ tiếp tục từ chỗ dừng"; exit 1; }

In [ ]:
%%bash
# A-6: gộp kết quả khứ hồi rồi tổng hợp bằng số đếm (số mẫu sinh, số mẫu được giữ theo số call, theo model viết câu) và ghi 30 dòng ngẫu nhiên của mẫu ĐƯỢC GIỮ. KHÔNG in nội dung câu.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
cat data/processed/chk_0*.jsonl > data/processed/callgen_checked_all.jsonl
wc -l data/processed/callgen_checked_all.jsonl
python -X utf8 /kaggle/working/helpers/a5g_summary.py data/processed/callgen_checked_all.jsonl data/processed/txt_8b.jsonl data/processed/txt_9b.jsonl /kaggle/working/a5g_out 2>&1 | tee outputs/logs/a5g_summary.txt
ls -la /kaggle/working/a5g_out

In [ ]:
%%bash
# A-7 (CPU, 1 đến 2 phút): XEM TRƯỚC số mẫu thật sự dùng được khi train theo bộ chia A4. Chạy đúng lệnh build_sft của Phương, chỉ để đọc dòng
# "callgen -> sft_train_cg.jsonl: N samples; {...}" (số chính xác, đã trừ mẫu có gốc/donor ngoài phần train của bộ chia, mẫu dùng app bị giấu, câu chép câu thật).
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 scripts/build_sft.py --asr-tag phowl --hold-apps flights,bike_rental,package_tracking,shopping_list,podcast_radio,loyalty --hist-max 300 \
  --callgen data/processed/callgen_checked_all.jsonl 2>&1 | cut -c1-400 | tee outputs/logs/a5g_build_preview.txt
echo; grep "callgen ->" outputs/logs/a5g_build_preview.txt || echo "KHÔNG thấy dòng callgen: báo lại nguyên văn log trên"
wc -l data/processed/holdout/sft_train.jsonl data/processed/holdout/eval_ids.txt data/processed/holdout/sft_train_cg.jsonl

In [ ]:
%%bash
# A-8: lưu dữ liệu cho notebook train (a5_data/callgen_checked_all.jsonl, KHÔNG có thư mục state/) và gói kết quả nhỏ để gửi Phương: a5_gen_results.zip
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
mkdir -p /kaggle/working/a5_data
cp data/processed/callgen_checked_all.jsonl /kaggle/working/a5_data/callgen_checked_all.jsonl
sha256sum /kaggle/working/a5_data/callgen_checked_all.jsonl | tee /kaggle/working/a5_data/callgen_checked_all.sha256
wc -l /kaggle/working/a5_data/callgen_checked_all.jsonl
python -X utf8 /kaggle/working/helpers/pack_zip.py /kaggle/working/a5_gen_results.zip \
  /kaggle/working/a5g_out/a5g_sample30_kept.txt /kaggle/working/a5g_out/a5g_sample30_kept.jsonl \
  outputs/logs/a5g_*.txt /kaggle/working/torch_cuda.txt /kaggle/working/gen_model_info.txt /kaggle/working/a5_data/callgen_checked_all.sha256
ls -la /kaggle/working/*.zip /kaggle/working/a5_data

In [ ]:
%%bash
# A-9 (CUỐI CÙNG): dọn thư mục làm việc (xóa lifemate/ có symlink tới dữ liệu đọc-chỉ và bản sao adapter), giữ a5_data/, zip và log. Rồi Save Version.
# KHÔNG chạy kaggle_save_state.sh: output này không được có thư mục state/ (nếu có, notebook train sẽ khôi phục nhầm).
cd /kaggle/working
rm -rf lifemate helpers setup_patched.sh config_paths.txt checkpoints state
du -sh /kaggle/working/* 2>/dev/null
ls /kaggle/working
echo "xong. Bây giờ Save Version (private). Lấy a5_gen_results.zip ở tab Output; notebook train sẽ gắn output này để lấy a5_data/callgen_checked_all.jsonl."